In [2]:
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix)

In [3]:
SEED = 42
df = pd.read_csv("H2.csv")

In [4]:
leaky = ["ReservationStatus", "ReservationStatusDate"]   # direct leakage
other = ["ADR", "AssignedRoomType", "Agent", "Company", "Country"]  # temporary columuns to be dropped some might be indirect leakage 
                                                                    # and some might be irrelevant to the target 
                                                                    # or would be alot for some dummy testing now
X = df.drop(columns=["IsCanceled"] + leaky + other) #drop leaky and other and the target column
y = df["IsCanceled"]

In [5]:
df.sample(5, random_state=42)

,IsCanceled,LeadTime,ArrivalDateYear,ArrivalDateMonth,ArrivalDateWeekNumber,ArrivalDateDayOfMonth,StaysInWeekendNights,StaysInWeekNights,Adults,Children,...,DepositType,Agent,Company,DaysInWaitingList,CustomerType,ADR,RequiredCarParkingSpaces,TotalOfSpecialRequests,ReservationStatus,ReservationStatusDate
13765,1,131,2016,June,27,30,0,3,2,1.0,...,No Deposit,9,NULL,0,Transient,112.50,0,0,Canceled,2016-02-20
78659,0,116,2017,August,34,21,1,3,1,0.0,...,No Deposit,98,NULL,0,Transient-Party,125.00,0,0,Check-Out,2017-08-25
51395,0,131,2016,June,25,17,2,3,2,0.0,...,No Deposit,9,NULL,0,Transient,105.74,0,1,Check-Out,2016-06-22
33999,1,314,2015,August,35,27,0,2,2,0.0,...,Non Refund,1,NULL,0,Contract,62.00,0,0,Canceled,2015-01-01
69024,0,90,2017,April,14,2,2,2,2,0.0,...,No Deposit,8,NULL,0,Transient,117.00,0,1,Check-Out,2017-04-06


In [6]:
#already is numeric values in dateset but run for safety
X["Children"] = pd.to_numeric(X["Children"], errors="coerce") #dataset info lists Children as text, not a number
#errors="coerce" means: if a value can’t be turned into a number ("NULL" or "NA"), replace it with NaN (a missing value) instead of crashing

In [8]:
X.sample(5, random_state=42)

,LeadTime,ArrivalDateYear,ArrivalDateMonth,ArrivalDateWeekNumber,ArrivalDateDayOfMonth,StaysInWeekendNights,StaysInWeekNights,Adults,Children,Babies,...,IsRepeatedGuest,PreviousCancellations,PreviousBookingsNotCanceled,ReservedRoomType,BookingChanges,DepositType,DaysInWaitingList,CustomerType,RequiredCarParkingSpaces,TotalOfSpecialRequests
13765,131,2016,June,27,30,0,3,2,1.0,0,...,0,0,0,A,0,No Deposit,0,Transient,0,0
78659,116,2017,August,34,21,1,3,1,0.0,0,...,0,0,0,D,2,No Deposit,0,Transient-Party,0,0
51395,131,2016,June,25,17,2,3,2,0.0,0,...,0,0,0,A,1,No Deposit,0,Transient,0,1
33999,314,2015,August,35,27,0,2,2,0.0,0,...,0,1,0,A,0,Non Refund,0,Contract,0,0
69024,90,2017,April,14,2,2,2,2,0.0,0,...,0,0,0,A,0,No Deposit,0,Transient,0,1


In [ ]:
y.sample(5, random_state=42)    

13765    1
78659    0
51395    0
33999    1
69024    0
Name: IsCanceled, dtype: int64

In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED)

In [ ]:
#defines the recipe (pre). It doesn’t change X The pipeline works on a copy internally.

# num = number columns (LeadTime, Adults, ...)
# cat = text columns (Meal, DepositType, ...)
num = X.select_dtypes("number").columns.tolist()
cat = X.select_dtypes(exclude="number").columns.tolist()

# SimpleImputer (median) fills missing values with the column’s middle value
# StandardScaler puts all columns on a similar scale (around 0)
#OneHotEncode It turns each category into its own 0/1 column.
pre = ColumnTransformer([
    ("num", Pipeline([("imp", SimpleImputer(strategy="median")),
                      ("sc", StandardScaler())]), num),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat),
])

#pre now has X but with all scaling and encoding done. It’s ready to be fed into a model.

In [13]:
pre

,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse matrices,these will be stacked as a sparse matrix if the overall density islower than this value. Use ``sparse_threshold=0`` to always returndense. When the transformed output consists of all dense data, thestacked result will be dense, and this keyword will be ignored.",0.3
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details.",None
,"transformer_weights transformer_weights: dict, default=NoneMultiplicative weights for features per transformer. The output of thetransformer is multiplied by these weights. Keys are transformer names,values the weights.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each transformer will beprinted as it is completed.",False
,"verbose_feature_names_out verbose_feature_names_out: bool, str or Callable[[str, str], str], default=True- If True, :meth:`ColumnTransformer.get_feature_names_out` will prefix all feature names with the name of the transformer that generated that feature. It is equivalent to setting `verbose_feature_names_out=""{transformer_name}__{feature_name}""`.- If False, :meth:`ColumnTransformer.get_feature_names_out` will not prefix any feature names and will error if feature names are not unique.- If ``Callable[[str, str], str]``, :meth:`ColumnTransformer.get_feature_names_out` will rename all the features using the name of the transformer. The first argument of the callable is the transformer name and the second argument is the feature name. The returned string will be the new feature name.- If ``str``, it must be a string ready for formatting. The given string will be formatted using two field names: ``transformer_name`` and ``feature_name``. e.g. `

In [14]:
pre.fit(X_train)
X_ready = pre.transform(X_train)

print(X_ready.shape)   # rows x new number of columns

pd.DataFrame(X_ready.toarray() if hasattr(X_ready, "toarray") else X_ready,
             columns=pre.get_feature_names_out()).head()

(63464, 60)


,num__LeadTime,num__ArrivalDateYear,num__ArrivalDateWeekNumber,num__ArrivalDateDayOfMonth,num__StaysInWeekendNights,num__StaysInWeekNights,num__Adults,num__Children,num__Babies,num__IsRepeatedGuest,...,cat__ReservedRoomType_F,cat__ReservedRoomType_G,cat__ReservedRoomType_P,cat__DepositType_No Deposit,cat__DepositType_Non Refund,cat__DepositType_Refundable,cat__CustomerType_Contract,cat__CustomerType_Group,cat__CustomerType_Transient,cat__CustomerType_Transient-Party
0,1.784280,-0.251414,1.557685,1.054201,-0.894336,-0.126248,0.288949,-0.246975,-0.055768,-0.163431,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0
1,-0.987752,-1.681652,0.586926,1.398043,-0.894336,-0.126248,0.288949,-0.246975,-0.055768,6.118802,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
2,0.614267,1.178824,0.064210,-0.321164,1.361015,1.917506,0.288949,5.077088,-0.055768,-0.163431,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
3,-0.285744,-0.251414,1.557685,0.824974,-0.894336,-0.807499,-1.671900,-0.246975,-0.055768,-0.163431,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
4,-0.438745,-1.681652,0.661600,-1.238074,1.361015,-0.126248,0.288949,-0.246975,-0.055768,-0.163431,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0


In [ ]:
#Baseline (untuned)
models = {
    "logreg": LogisticRegression(max_iter=1000),
    "rf": RandomForestClassifier(n_estimators=100, n_jobs=-1, random_state=SEED),
    "xgb": XGBClassifier(random_state=SEED, eval_metric="logloss"),
}

cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
from sklearn.model_selection import cross_val_score

baseline = {} #empty list to store the results of the models
for name, m in models.items():
    pipe = Pipeline([("pre", pre), ("model", m)])
    #F1 measures how correct the model is. It goes from 0 to 1
    baseline[name] = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="f1").mean()
    print(name, round(baseline[name], 4))

logreg 0.7319
rf 0.8162
xgb 0.7966


In [17]:
#Tune hyperparameters

# A dictionary: model name → settings to try, with a list of allowed values for each.
# The model__ prefix is needed because the model sits inside a pipeline step named "model".
# It means “this setting belongs to the step called model”
param_grids = {
    "logreg": {"model__C": [0.01, 0.1, 1, 10, 100]},
    "rf": {"model__n_estimators": [100, 200, 300],
           "model__max_depth": [10, 20, 30, None],
           "model__min_samples_leaf": [1, 2, 5],
           "model__max_features": ["sqrt", 0.3]},
    "xgb": {"model__n_estimators": [200, 400],
            "model__max_depth": [4, 6, 8],
            "model__learning_rate": [0.03, 0.1, 0.2],
            "model__subsample": [0.7, 1.0],
            "model__colsample_bytree": [0.7, 1.0]},
}

best = {}
for name, m in models.items():
    pipe = Pipeline([("pre", pre), ("model", m)])
    search = RandomizedSearchCV(pipe, param_grids[name], n_iter=10, cv=cv,
                                scoring="f1", n_jobs=-1, random_state=SEED)
    search.fit(X_train, y_train)
    best[name] = search.best_estimator_
    print(name, search.best_params_, round(search.best_score_, 4))

c:\Users\DELL\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\model_selection\_search.py:324: UserWarning: The total space of parameters 5 is smaller than n_iter=10. Running 5 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


logreg {'model__C': 1} 0.7319
rf {'model__n_estimators': 100, 'model__min_samples_leaf': 1, 'model__max_features': 'sqrt', 'model__max_depth': None} 0.8162
xgb {'model__subsample': 0.7, 'model__n_estimators': 400, 'model__max_depth': 8, 'model__learning_rate': 0.2, 'model__colsample_bytree': 0.7} 0.8105


In [18]:
rows = []
for name, est in best.items():
    pred = est.predict(X_test)
    rows.append({"model": name,
                 "accuracy": accuracy_score(y_test, pred),
                 "precision": precision_score(y_test, pred),
                 "recall": recall_score(y_test, pred),
                 "f1": f1_score(y_test, pred),
                 "roc_auc": roc_auc_score(y_test, est.predict_proba(X_test)[:, 1])})
    print(name, "\n", confusion_matrix(y_test, pred))
pd.DataFrame(rows).round(4)

logreg 
 [[8529  717]
 [2363 4257]]
rf 
 [[8465  781]
 [1455 5165]]
xgb 
 [[8377  869]
 [1466 5154]]


,model,accuracy,precision,recall,f1,roc_auc
0,logreg,0.8059,0.8559,0.6431,0.7343,0.8558
1,rf,0.8591,0.8687,0.7802,0.8221,0.9270
2,xgb,0.8528,0.8557,0.7785,0.8153,0.9238
